# AI-Lake + Apache Spark

Spark reads AI-Lake tables as standard Iceberg tables — no AI-Lake plugin required
for tabular queries. This notebook uses **PySpark local[*]** mode, which runs
entirely inside the Jupyter container (no separate Spark cluster needed).

**What this notebook covers:**
1. Single SparkSession with Iceberg JAR pre-loaded
2. Direct Parquet read (plain binary for `embedding`)
3. Iceberg HadoopCatalog SQL — full SQL interface
4. Filtered scans and aggregations
5. Iceberg snapshot history

In [ ]:
import os
import pathlib
from pyspark.sql import SparkSession

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
ICEBERG_JAR = '/opt/iceberg/iceberg-spark-runtime-3.5_2.12-1.5.2.jar'

parquet_files = sorted(str(p) for p in pathlib.Path(TABLE_PATH).rglob('*.parquet'))
print(f'Table path   : {TABLE_PATH}')
print(f'Parquet files: {len(parquet_files)}')
print(f'Iceberg JAR  : {ICEBERG_JAR}')

## 1. Create SparkSession (Iceberg pre-loaded)

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

# Single session — Iceberg JAR loaded from the start so catalog plugin
# is on the JVM classpath when the session is created.
spark = (
    SparkSession.builder
    .appName('ailake-spark-demo')
    .master('local[2]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.jars', ICEBERG_JAR)
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions',
    )
    .config('spark.sql.catalog.ailake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.ailake.type', 'hadoop')
    .config('spark.sql.catalog.ailake.warehouse', f'file://{TABLE_PATH}')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} — local[2] with Iceberg extensions')

## 2. Direct Parquet read

In [ ]:
df = spark.read.parquet(*parquet_files)
print(f'Rows: {df.count()}')
print('Schema:')
df.printSchema()

In [ ]:
# embedding shows as binary — HNSW footer is invisible to Spark
df.select('text').show(5, truncate=80)

## 3. Iceberg HadoopCatalog (SQL interface)

In [ ]:
# Same session — catalog plugin already loaded on classpath
count = spark.sql(
    'SELECT count(*) AS n FROM ailake.default.table'
).collect()[0][0]
print(f'Iceberg SQL count: {count}')

In [ ]:
spark.sql("""
    SELECT text
    FROM ailake.default.table
    WHERE text LIKE '%vector search%'
    LIMIT 5
""").show(truncate=90)

## 4. Aggregations

In [ ]:
spark.sql("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_len,
        avg(octet_length(embedding)) AS avg_emb_bytes
    FROM ailake.default.table
""").show()

## 5. Iceberg snapshot history

In [ ]:
spark.sql(
    'SELECT snapshot_id, committed_at, operation, summary'
    ' FROM ailake.default.table.snapshots'
).show(truncate=False)

In [ ]:
spark.stop()
print('SparkSession stopped.')

## Key takeaway

Spark reads AI-Lake tables as **standard Iceberg** — the `HadoopCatalog` discovers
the Parquet files from `metadata.json`, and the Iceberg Spec v2 manifests are fully
compatible. No AI-Lake plugin required for tabular queries.

The `embedding` column appears as `binary` — Spark reads the raw F16 bytes but does
not interpret them. For vector similarity search, use the AI-Lake SDK directly
(`ailake.search()` — see `01_ailake_demo.ipynb`) or the Spark plugin.

> **Note (dim=32):** The demo fixture uses dim=32 embeddings. Each vector column
> row is 64 bytes (`dim × 2` for F16 storage).